# 05. 발신 통화 분류 결과 시각화

`03_analyze_outbound_calls.ipynb`의 분류 결과(`../data/outbound/outbound_call_classification.json`)를 읽어
두 가지 관점으로 16개의 분석 이미지를 생성합니다.

- **Part A. 통화 유형(`call_type`) 기반 분석** — 고객콜백/공급업체/기타 분류를 그대로 사용해 콜백 이유, 해결 상태, 고객 감정, QA 패턴, 공급업체 발주·재고·이슈를 분석합니다.
- **Part B. 통화 목적·수신빈도 기반 분석** — `call_type` 분류와 무관하게, 전사문 키워드로 재분류한 통화 목적과 수신번호 연락 빈도를 축으로 분석합니다. `call_type`이 놓칠 수 있는 실질적 업무 패턴(어떤 번호와 왜 자주 연락하는지)을 보완합니다.

- **입력**: `../data/outbound/outbound_call_classification.json`
- **출력**: `../output/outbound/01_call_type_overview.png` ~ `16_comprehensive_insight.png` (16개)

In [ ]:
import os
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

warnings.filterwarnings("ignore")

sns.set_style("whitegrid", {"font.family": ["Malgun Gothic"]})
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.facecolor"] = "white"

TYPE_COLORS = {"고객콜백": "#2196F3", "공급업체": "#4CAF50", "기타": "#9E9E9E"}
RES_COLORS = {"해결": "#4CAF50", "부분해결": "#FF9800", "미해결": "#F44336", "불명확": "#9E9E9E"}
SENT_COLORS = {"만족": "#2196F3", "중립": "#9E9E9E", "불만": "#F44336", "불명확": "#BDBDBD"}
CAT_COLORS = [
    "#E91E63", "#2196F3", "#4CAF50", "#FF9800", "#9C27B0",
    "#00BCD4", "#FF5722", "#795548", "#607D8B", "#9E9E9E",
]
TOPIC_COLORS = {
    "인쇄/제작 확인": "#E91E63",
    "납기/배송 확인": "#4CAF50",
    "재고/수량 확인": "#2196F3",
    "가격/단가 협의": "#FF9800",
    "주문/발주 처리": "#9C27B0",
    "상품 사양 문의": "#00BCD4",
    "기타": "#9E9E9E",
}
TOPIC_ORDER = list(TOPIC_COLORS.keys())

DATA_PATH = "../data/outbound/outbound_call_classification.json"
OUTPUT_DIR = "../output/outbound"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("환경 설정 완료")

In [ ]:
# ============================================================
# 데이터 로드 & 공통 전처리
# ============================================================
with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

df = pd.DataFrame(raw)
df["folder"] = df["folder"].astype(str)
df["month"] = df["folder"].str[:6]
df["year_month"] = pd.to_datetime(df["month"], format="%Y%m")
df["qa_count"] = df["qa_pairs"].apply(lambda x: len(x) if isinstance(x, list) else 0)
df["product_count"] = df["products"].apply(lambda x: len(x) if isinstance(x, list) else 0)
df["duration_min"] = df["duration_sec"] / 60


def extract_receiver(fname):
    parts = str(fname).split("_")
    return parts[1] if len(parts) > 1 else ""


def extract_hour(fname):
    try:
        parts = str(fname).split("_")
        if len(parts) >= 7:
            return int(parts[5])
    except Exception:
        pass
    return None


def extract_weekday(fname):
    try:
        parts = str(fname).split("_")
        if len(parts) >= 8:
            dt = datetime.strptime("_".join(parts[2:8]), "%Y_%m_%d_%H_%M_%S")
            return dt.weekday()
    except Exception:
        pass
    return None


df["receiver"] = df["file_name"].apply(extract_receiver)
df["hour"] = df["file_name"].apply(extract_hour)
df["weekday"] = df["file_name"].apply(extract_weekday)

# call_type 기반 서브셋 (Part A) — 03의 스키마는 label 없이 고객콜백 전체가 상담 대상
cb_df = df[df["call_type"] == "고객콜백"].copy()
sup_df = df[df["call_type"] == "공급업체"].copy()
etc_df = df[df["call_type"] == "기타"].copy()

all_qa = []
for _, row in cb_df.iterrows():
    for qa in (row.get("qa_pairs") or []):
        if isinstance(qa, dict) and qa.get("question") and qa.get("answer"):
            all_qa.append({
                "month": row["month"],
                "duration_sec": row["duration_sec"],
                "callback_reason": row.get("callback_reason", ""),
                "question": qa.get("question", ""),
                "answer": qa.get("answer", ""),
                "q_len": len(qa.get("question", "")),
                "a_len": len(qa.get("answer", "")),
            })
qa_df = pd.DataFrame(all_qa)

all_products = []
for _, row in sup_df.iterrows():
    for p in (row.get("products") or []):
        if isinstance(p, dict) and p.get("product_name"):
            all_products.append({
                "month": row["month"],
                "supplier_name": row.get("supplier_name", ""),
                "product_name": p.get("product_name", ""),
                "quantity": p.get("quantity"),
                "unit_price": p.get("unit_price"),
                "stock_available": p.get("stock_available"),
                "delivery_date": p.get("delivery_date", ""),
                "print_condition": p.get("print_condition", ""),
                "notes": p.get("notes", ""),
            })
prod_df = pd.DataFrame(all_products)

print("=" * 55)
print("발신 통화 데이터 로드 완료")
print("=" * 55)
print(f"전체 발신 통화:   {len(df):>5,}건")
print(f"고객콜백:         {len(cb_df):>5,}건")
print(f"공급업체:         {len(sup_df):>5,}건")
print(f"기타:             {len(etc_df):>5,}건")
print(f"분석 기간:        {df.month.min()} ~ {df.month.max()}")
print(f"고객콜백 QA 쌍:   {len(qa_df):>5,}개")
print(f"공급업체 상품 언급: {len(prod_df):>4,}건")

---
# Part A. 통화 유형(`call_type`) 기반 분석

## 1. 발신 통화 전체 현황

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle("기프트코 발신 통화 전체 현황", fontsize=16, fontweight="bold")

# 1. call_type 파이
ax1 = axes[0, 0]
tc = df["call_type"].value_counts()
colors_pie = [TYPE_COLORS.get(t, "#9E9E9E") for t in tc.index]
wedges, texts, autotexts = ax1.pie(
    tc.values, labels=tc.index, autopct="%1.1f%%",
    colors=colors_pie, startangle=90,
    wedgeprops=dict(edgecolor="white", linewidth=2),
)
for at in autotexts:
    at.set_fontsize(10)
    at.set_fontweight("bold")
ax1.set_title(f"통화 유형 분포 (총 {len(df):,}건)", fontweight="bold", pad=10)

# 2. 월별 유형 누적
ax2 = axes[0, 1]
type_order = [t for t in ["고객콜백", "공급업체", "기타"] if t in df["call_type"].unique()]
monthly_type = df.groupby(["month", "call_type"]).size().unstack(fill_value=0)
monthly_type = monthly_type.reindex(columns=type_order, fill_value=0)
x = range(len(monthly_type))
bottom = np.zeros(len(monthly_type))
for col in type_order:
    ax2.bar(x, monthly_type[col], bottom=bottom,
            color=TYPE_COLORS.get(col, "#9E9E9E"), label=col, alpha=0.85)
    bottom += monthly_type[col].values
ax2.set_xticks(list(x))
ax2.set_xticklabels([m[2:] for m in monthly_type.index], rotation=45, fontsize=8)
ax2.set_title("월별 통화 유형 누적", fontweight="bold")
ax2.set_ylabel("건수")
ax2.legend(fontsize=8, loc="upper left")

# 3. 통화시간 박스플롯
ax3 = axes[1, 0]
data_box = [df[df["call_type"] == t]["duration_sec"].clip(0, 600).values for t in type_order]
bp = ax3.boxplot(data_box, labels=type_order, patch_artist=True,
                  medianprops=dict(color="black", linewidth=2))
for patch, t in zip(bp["boxes"], type_order):
    patch.set_facecolor(TYPE_COLORS.get(t, "#9E9E9E"))
    patch.set_alpha(0.7)
ax3.set_title("통화 유형별 통화시간 (초, 600초 상한)", fontweight="bold")
ax3.set_ylabel("통화시간 (초)")
for i, data in enumerate(data_box, 1):
    if len(data) > 0:
        ax3.text(i, np.mean(data) + 10, f"평균\n{np.mean(data):.0f}초",
                  ha="center", fontsize=8, color="darkblue")

# 4. 월별 고객콜백 QA 수
ax4 = axes[1, 1]
if len(qa_df) > 0:
    monthly_qa = qa_df.groupby("month").size().reset_index(name="qa수")
    ax4.bar(range(len(monthly_qa)), monthly_qa["qa수"],
            color="#2196F3", alpha=0.8, edgecolor="white")
    ax4.set_xticks(range(len(monthly_qa)))
    ax4.set_xticklabels([m[2:] for m in monthly_qa["month"]], rotation=45, fontsize=8)
    for i, v in enumerate(monthly_qa["qa수"]):
        ax4.text(i, v + 0.5, str(v), ha="center", fontsize=8)
ax4.set_title(f"월별 고객콜백 QA 생성 건수 (총 {len(qa_df):,}개)", fontweight="bold")
ax4.set_ylabel("QA 수")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/01_call_type_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print("저장: 01_call_type_overview.png")

## 2. 고객콜백 — 콜백 이유 · 해결 상태 · 고객 감정

In [ ]:
# callback_reason 자동 카테고리화
REASON_KEYWORDS = {
    "배송/출고 안내": ["배송", "출고", "도착", "택배", "발송", "납기", "배달"],
    "견적/가격 답변": ["견적", "가격", "단가", "비용", "금액", "얼마"],
    "주문/발주 확인": ["주문", "발주", "오더", "확인", "접수"],
    "결제/세금계산서": ["결제", "세금계산서", "영수증", "입금", "계좌", "카드"],
    "인쇄/디자인 안내": ["인쇄", "디자인", "파일", "로고", "시안", "제작", "이미지"],
    "샘플 관련": ["샘플", "견본", "시제품"],
    "취소/환불": ["취소", "환불", "반품"],
    "사이트/계정": ["로그인", "아이디", "비밀번호", "회원", "사이트"],
    "재고/상품 안내": ["재고", "상품", "제품", "코드"],
}


def classify_reason(text):
    text = str(text or "").lower()
    scores = {}
    for cat, kws in REASON_KEYWORDS.items():
        s = sum(text.count(kw) for kw in kws)
        if s > 0:
            scores[cat] = s
    return max(scores, key=scores.get) if scores else "기타"


cb_df["reason_cat"] = cb_df["callback_reason"].apply(classify_reason)
reason_counts = cb_df["reason_cat"].value_counts()
total_cb = len(cb_df)

print("[ 고객콜백 — 콜백 이유 자동 분류 ]")
for cat, cnt in reason_counts.items():
    bar = "█" * int(cnt / total_cb * 30)
    print(f"  {cat:<18}: {cnt:>4}건 ({cnt/total_cb*100:5.1f}%)  {bar}")
print(f"  {'합계':<18}: {total_cb:>4}건")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("고객콜백 — 콜백 이유 · 해결 상태 · 고객 감정", fontsize=14, fontweight="bold")

# 1. 콜백 이유 수평 막대
ax1 = axes[0]
r_list = reason_counts.index.tolist()
r_vals = reason_counts.values.tolist()
colors_r = CAT_COLORS[:len(r_list)]
bars1 = ax1.barh(r_list[::-1], r_vals[::-1], color=colors_r[::-1], alpha=0.85, edgecolor="white")
for bar, val in zip(bars1, r_vals[::-1]):
    ax1.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
              f"{val}건 ({val/total_cb*100:.1f}%)", va="center", fontsize=8)
ax1.set_xlim(0, max(r_vals) * 1.35)
ax1.set_title("콜백 이유 분류", fontweight="bold")
ax1.set_xlabel("건수")

# 2. 해결 상태
ax2 = axes[1]
if "resolution_status" in cb_df.columns:
    res = cb_df["resolution_status"].value_counts()
    res_order = [r for r in ["해결", "부분해결", "미해결", "불명확"] if r in res.index]
    res_vals = [res.get(r, 0) for r in res_order]
    colors_res = [RES_COLORS.get(r, "#9E9E9E") for r in res_order]
    wedges, texts, auts = ax2.pie(
        res_vals, labels=res_order, autopct="%1.1f%%",
        colors=colors_res, startangle=90,
        wedgeprops=dict(edgecolor="white", linewidth=2),
    )
    for at in auts:
        at.set_fontsize(9)
        at.set_fontweight("bold")
    solved = res.get("해결", 0) + res.get("부분해결", 0)
    ax2.set_title(f"해결 상태 (해결+부분={solved/total_cb*100:.0f}%)", fontweight="bold")

# 3. 고객 감정
ax3 = axes[2]
if "customer_sentiment" in cb_df.columns:
    sent = cb_df["customer_sentiment"].value_counts()
    sent_order = [s for s in ["만족", "중립", "불만", "불명확"] if s in sent.index]
    sent_vals = [sent.get(s, 0) for s in sent_order]
    colors_sent = [SENT_COLORS.get(s, "#9E9E9E") for s in sent_order]
    bars3 = ax3.bar(sent_order, sent_vals, color=colors_sent, edgecolor="white", alpha=0.85)
    for bar, val in zip(bars3, sent_vals):
        ax3.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                  f"{val}\n({val/total_cb*100:.1f}%)", ha="center", fontsize=9, fontweight="bold")
    ax3.set_title("고객 감정 분포", fontweight="bold")
    ax3.set_ylabel("건수")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/02_callback_reason_resolution_sentiment.png", dpi=150, bbox_inches="tight")
plt.show()
print("저장: 02_callback_reason_resolution_sentiment.png")

In [ ]:
# 콜백 이유별 × 해결 상태 크로스탭
if "resolution_status" in cb_df.columns:
    cross = pd.crosstab(cb_df["reason_cat"], cb_df["resolution_status"])
    cross_pct = cross.div(cross.sum(axis=1), axis=0) * 100

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle("콜백 이유별 해결 상태 교차 분석", fontsize=13, fontweight="bold")

    ax1 = axes[0]
    sns.heatmap(cross, annot=True, fmt="d", cmap="Blues", ax=ax1,
                linewidths=0.5, cbar_kws={"label": "건수"})
    ax1.set_title("절대 건수", fontweight="bold")
    ax1.set_xlabel("해결 상태")
    ax1.set_ylabel("콜백 이유")
    ax1.tick_params(axis="x", rotation=30)

    ax2 = axes[1]
    sns.heatmap(cross_pct.round(1), annot=True, fmt=".1f", cmap="RdYlGn", ax=ax2,
                linewidths=0.5, cbar_kws={"label": "%"})
    ax2.set_title("행 비율 (%)", fontweight="bold")
    ax2.set_xlabel("해결 상태")
    ax2.set_ylabel("")
    ax2.tick_params(axis="x", rotation=30)

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/03_callback_reason_resolution_cross.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("저장: 03_callback_reason_resolution_cross.png")

In [ ]:
# 후속 조치 분석 (텍스트, 이미지 없음)
if "follow_up_needed" in cb_df.columns:
    n_needed = (cb_df["follow_up_needed"] == True).sum()
    print("[ 후속 조치 필요 현황 ]")
    print(f"  필요:   {n_needed}건 ({n_needed/total_cb*100:.1f}%)")
    print(f"  불필요: {(cb_df['follow_up_needed']==False).sum()}건")
    print(f"  미기재: {cb_df['follow_up_needed'].isna().sum()}건")
    print()

    needs_fu = cb_df[cb_df["follow_up_needed"] == True]
    print("[ 후속 조치 필요 건 — 이유별 분류 ]")
    for cat, cnt in needs_fu["reason_cat"].value_counts().items():
        print(f"  {cat:<18}: {cnt}건")

    print("\n[ 후속 조치 내용 샘플 10개 ]")
    for _, row in needs_fu.head(10).iterrows():
        callback_reason = row.get("callback_reason")
        follow_up_detail = row.get("follow_up_detail")
        callback_reason = "" if pd.isna(callback_reason) else str(callback_reason)
        follow_up_detail = "" if pd.isna(follow_up_detail) else str(follow_up_detail)
        print(f"  콜백이유: {callback_reason[:50]}")
        print(f"  조치내용: {follow_up_detail[:80]}")
        print()

## 3. 고객콜백 — QA 패턴 분석

In [ ]:
if len(qa_df) == 0:
    print("QA 데이터 없음")
else:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle("고객콜백 QA 패턴 분석", fontsize=14, fontweight="bold")

    # 1. 질문/답변 길이 분포
    ax1 = axes[0, 0]
    ax1.hist(qa_df["q_len"].clip(0, 200), bins=30, color="#2196F3", alpha=0.7, label="질문")
    ax1.hist(qa_df["a_len"].clip(0, 200), bins=30, color="#4CAF50", alpha=0.5, label="답변")
    ax1.axvline(qa_df["q_len"].mean(), color="#1565C0", linestyle="--", linewidth=2,
                label=f"질문 평균 {qa_df['q_len'].mean():.0f}자")
    ax1.axvline(qa_df["a_len"].mean(), color="#2E7D32", linestyle="--", linewidth=2,
                label=f"답변 평균 {qa_df['a_len'].mean():.0f}자")
    ax1.set_title("QA 질문/답변 길이 분포", fontweight="bold")
    ax1.set_xlabel("글자 수")
    ax1.set_ylabel("빈도")
    ax1.legend(fontsize=8)

    # 2. 질문 유형 (콜백 맥락 반영)
    ax2 = axes[0, 1]
    q_types = {
        "가능/여부 확인": ["가능", "됩니까", "되나요", "돼요"],
        "언제 (배송/출고)": ["언제", "며칠", "몇 일", "몇일"],
        "얼마 (가격)": ["얼마", "가격이", "단가", "비용"],
        "있나요 (재고)": ["있나요", "있습니까", "재고"],
        "변경/수정 요청": ["변경", "수정", "바꿔", "바꿀"],
        "확인 요청": ["확인", "맞나요", "맞죠"],
        "이전 문의 재확인": ["아까", "전에", "지난번", "저번에", "이전에"],
        "결제/세금계산서": ["결제", "세금계산서", "영수증", "입금"],
    }
    q_type_counts = {}
    for qtype, kws in q_types.items():
        cnt = sum(qa_df["question"].str.contains(kw, na=False).sum() for kw in kws)
        if cnt > 0:
            q_type_counts[qtype] = cnt
    q_sorted = sorted(q_type_counts.items(), key=lambda x: -x[1])
    if q_sorted:
        q_names = [q for q, v in q_sorted]
        q_values = [v for q, v in q_sorted]
        bars2 = ax2.bar(q_names, q_values, color=CAT_COLORS[:len(q_names)], alpha=0.85, edgecolor="white")
        for bar, val in zip(bars2, q_values):
            ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                      str(val), ha="center", fontsize=9)
        ax2.set_title("고객 질문 유형 (콜백 맥락)", fontweight="bold")
        ax2.set_ylabel("QA 수")
        ax2.tick_params(axis="x", rotation=35, labelsize=8)

    # 3. 콜백 이유별 QA 수
    ax3 = axes[1, 0]

    def cat_reason(text):
        for cat, kws in REASON_KEYWORDS.items():
            if any(kw in str(text).lower() for kw in kws):
                return cat
        return "기타"

    qa_df["reason_cat"] = qa_df["callback_reason"].apply(cat_reason)
    qa_by_cat = qa_df.groupby("reason_cat").size().sort_values(ascending=False)
    if len(qa_by_cat) > 0:
        bars3 = ax3.barh(qa_by_cat.index[::-1], qa_by_cat.values[::-1],
                          color=CAT_COLORS[:len(qa_by_cat)], alpha=0.85, edgecolor="white")
        for bar, val in zip(bars3, qa_by_cat.values[::-1]):
            ax3.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                      str(val), va="center", fontsize=9)
        ax3.set_title("콜백 이유 카테고리별 QA 수", fontweight="bold")
        ax3.set_xlabel("QA 수")

    # 4. 답변 길이 구간
    ax4 = axes[1, 1]
    a_bins = [0, 30, 60, 100, 150, float("inf")]
    a_labels = ["~30자", "30~60자", "60~100자", "100~150자", "150자+"]
    qa_df["a_band"] = pd.cut(qa_df["a_len"], bins=a_bins, labels=a_labels)
    a_dist = qa_df["a_band"].value_counts().reindex(a_labels, fill_value=0)
    colors_a = ["#BBDEFB", "#90CAF9", "#64B5F6", "#42A5F5", "#1E88E5"]
    bars4 = ax4.bar(a_labels, a_dist.values, color=colors_a, edgecolor="white")
    for bar, val in zip(bars4, a_dist.values):
        if val > 0:
            ax4.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                      f"{val}개\n({val/len(qa_df)*100:.0f}%)", ha="center", fontsize=8)
    ax4.set_title("챗봇 답변 길이 분포", fontweight="bold")
    ax4.set_xlabel("답변 글자 수")
    ax4.set_ylabel("QA 수")

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/04_qa_patterns.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("저장: 04_qa_patterns.png")

## 4. 공급업체 — 재고·발주·이슈 분석

In [ ]:
if len(sup_df) == 0:
    print("공급업체 통화 데이터 없음")
else:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle("공급업체 통화 — 발주·재고·이슈 분석", fontsize=14, fontweight="bold")

    # 1. 발주 확정 현황
    ax1 = axes[0, 0]
    confirmed = (sup_df["order_confirmed"] == True).sum()
    unconfirmed = (sup_df["order_confirmed"] == False).sum()
    unknown = sup_df["order_confirmed"].isna().sum()
    oc_vals = [confirmed, unconfirmed, unknown]
    oc_labels = [f"확정 ({confirmed}건)", f"미확정 ({unconfirmed}건)", f"미기재 ({unknown}건)"]
    oc_colors = ["#4CAF50", "#F44336", "#9E9E9E"]
    ax1.pie(oc_vals, labels=oc_labels, autopct="%1.1f%%", colors=oc_colors,
            startangle=90, wedgeprops=dict(edgecolor="white", linewidth=2))
    ax1.set_title(f"발주 확정 현황 (총 {len(sup_df)}건)", fontweight="bold")

    # 2. 재고 여부 (상품 단위)
    ax2 = axes[0, 1]
    if len(prod_df) > 0:
        stock_map = {True: "재고있음", False: "재고없음", None: "미확인"}
        prod_df["stock_label"] = prod_df["stock_available"].map(lambda x: stock_map.get(x, "미확인"))
        stock_cnt = prod_df["stock_label"].value_counts()
        st_colors = {"재고있음": "#4CAF50", "재고없음": "#F44336", "미확인": "#9E9E9E"}
        bars2 = ax2.bar(stock_cnt.index, stock_cnt.values,
                         color=[st_colors.get(k, "#9E9E9E") for k in stock_cnt.index],
                         edgecolor="white", alpha=0.85)
        for bar, val in zip(bars2, stock_cnt.values):
            ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                      f"{val}건\n({val/len(prod_df)*100:.1f}%)", ha="center", fontsize=9, fontweight="bold")
        ax2.set_title(f"상품별 재고 여부 (총 {len(prod_df)}건)", fontweight="bold")
        ax2.set_ylabel("상품 언급 수")

    # 3. 이슈 있는 통화 월별 추세
    ax3 = axes[1, 0]
    monthly_sup = sup_df.groupby("month").agg(
        통화수=("call_type", "count"),
        발주확정=("order_confirmed", lambda x: (x == True).sum()),
        이슈건수=("issues", lambda x: x.apply(bool).sum()),
    ).reset_index()
    x_s = range(len(monthly_sup))
    ax3.bar(x_s, monthly_sup["통화수"], color="#A5D6A7", label="전체", alpha=0.7)
    ax3.bar(x_s, monthly_sup["발주확정"], color="#4CAF50", label="발주확정", alpha=0.9)
    ax3.bar(x_s, monthly_sup["이슈건수"], color="#F44336", label="이슈", alpha=0.9)
    ax3.set_xticks(list(x_s))
    ax3.set_xticklabels([m[2:] for m in monthly_sup["month"]], rotation=45, fontsize=8)
    ax3.set_title("월별 공급업체 통화 · 발주 · 이슈", fontweight="bold")
    ax3.set_ylabel("건수")
    ax3.legend(fontsize=8)

    # 4. 이슈 유형 분류
    ax4 = axes[1, 1]
    ISSUE_KEYWORDS = {
        "재고 없음": ["재고 없", "품절", "단종", "없어요", "없습니다"],
        "납기 지연": ["지연", "늦어", "늦을", "일정 변경", "납기"],
        "가격 이견": ["가격", "단가", "비싸", "조정", "협의"],
        "최소수량": ["최소", "MOQ", "moq", "수량 제한"],
        "품질/사양": ["품질", "사양", "규격", "불량"],
    }
    issue_texts = " ".join(sup_df["issues"].dropna().astype(str))
    issue_type_counts = {}
    for it, kws in ISSUE_KEYWORDS.items():
        cnt = sum(issue_texts.count(kw) for kw in kws)
        if cnt > 0:
            issue_type_counts[it] = cnt

    if issue_type_counts:
        it_sorted = sorted(issue_type_counts.items(), key=lambda x: -x[1])
        it_names = [k for k, v in it_sorted]
        it_values = [v for k, v in it_sorted]
        bars4 = ax4.barh(it_names[::-1], it_values[::-1],
                          color=["#F44336", "#FF9800", "#FF5722", "#795548", "#9E9E9E"][:len(it_names)],
                          alpha=0.85, edgecolor="white")
        for bar, val in zip(bars4, it_values[::-1]):
            ax4.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
                      str(val), va="center", fontsize=9)
        ax4.set_title("이슈 유형 분류 (키워드 기반)", fontweight="bold")
        ax4.set_xlabel("언급 횟수")
    else:
        ax4.text(0.5, 0.5, "이슈 데이터 없음", ha="center", va="center",
                  transform=ax4.transAxes, fontsize=12)

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/05_supplier_order_stock_issue.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("저장: 05_supplier_order_stock_issue.png")

In [ ]:
if len(prod_df) > 0:
    PROD_KEYWORDS = {
        "볼펜/펜": ["볼펜", "펜", "형광펜", "사인펜"],
        "머그/텀블러": ["머그", "텀블러", "컵"],
        "가방/에코백": ["가방", "에코백", "파우치", "백팩", "토트"],
        "의류": ["티셔츠", "점퍼", "후드", "조끼", "유니폼"],
        "인쇄물": ["인쇄물", "브로셔", "현수막", "명함", "스티커"],
        "우산": ["우산", "양산"],
        "노트/다이어리": ["노트", "다이어리", "수첩"],
        "보조배터리": ["배터리", "충전기", "보조배터리"],
        "수건/타올": ["수건", "타올", "타월"],
        "달력": ["달력", "캘린더"],
        "USB": ["usb", "USB"],
    }

    def classify_product(name):
        name_l = str(name).lower()
        for cat, kws in PROD_KEYWORDS.items():
            if any(kw.lower() in name_l for kw in kws):
                return cat
        return "기타"

    prod_df["prod_cat"] = prod_df["product_name"].apply(classify_product)
    prod_cat_cnt = prod_df["prod_cat"].value_counts()

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle("공급업체 통화 — 언급 상품 분석", fontsize=13, fontweight="bold")

    ax1 = axes[0]
    colors_prod = plt.cm.Set3(np.linspace(0, 1, len(prod_cat_cnt)))
    bars1 = ax1.barh(prod_cat_cnt.index[::-1], prod_cat_cnt.values[::-1],
                      color=colors_prod, edgecolor="white")
    for bar, val in zip(bars1, prod_cat_cnt.values[::-1]):
        ax1.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
                  f"{val}건", va="center", fontsize=9)
    ax1.set_title("공급업체 통화에서 언급된 상품 카테고리", fontweight="bold")
    ax1.set_xlabel("언급 건수")

    ax2 = axes[1]
    stock_by_cat = prod_df.groupby(["prod_cat", "stock_label"]).size().unstack(fill_value=0)
    cols_to_plot = [c for c in ["재고있음", "재고없음", "미확인"] if c in stock_by_cat.columns]
    if cols_to_plot:
        stock_by_cat[cols_to_plot].plot(
            kind="barh", ax=ax2,
            color=[{"재고있음": "#4CAF50", "재고없음": "#F44336", "미확인": "#9E9E9E"}.get(c, "#9E9E9E") for c in cols_to_plot],
            alpha=0.85, edgecolor="white",
        )
    ax2.set_title("카테고리별 재고 현황", fontweight="bold")
    ax2.set_xlabel("상품 수")
    ax2.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/06_supplier_product_analysis.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("저장: 06_supplier_product_analysis.png")

## 5. 시간대별 발신 패턴 (유형별)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("시간대별 발신 통화 패턴", fontsize=14, fontweight="bold")

all_hours = list(range(8, 20))
hour_colors = [
    "#FFCDD2" if h < 9 or h >= 18 else
    "#90CAF9" if 9 <= h < 12 else
    "#A5D6A7" if 12 <= h < 14 else "#42A5F5"
    for h in all_hours
]

# 1. 전체 시간대별
ax1 = axes[0, 0]
h_all = df[df["hour"].notna()]["hour"].value_counts().sort_index()
h_full = pd.Series([h_all.get(h, 0) for h in all_hours], index=all_hours)
bars1 = ax1.bar([f"{h}시" for h in all_hours], h_full.values, color=hour_colors, edgecolor="white")
for bar, val in zip(bars1, h_full.values):
    if val > 0:
        ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3, str(val), ha="center", fontsize=8)
ax1.set_title("전체 발신 — 시간대별 건수", fontweight="bold")
ax1.set_ylabel("건수")
ax1.tick_params(axis="x", rotation=45, labelsize=8)

# 2. 유형별 시간대
ax2 = axes[0, 1]
for t, color in [("고객콜백", "#2196F3"), ("공급업체", "#4CAF50")]:
    sub = df[(df["call_type"] == t) & df["hour"].notna()]
    h_sub = sub["hour"].value_counts().sort_index()
    h_sub_full = pd.Series([h_sub.get(h, 0) for h in all_hours], index=all_hours)
    ax2.plot(all_hours, h_sub_full.values, "o-", color=color, label=t, linewidth=2, markersize=6)
ax2.set_xticks(all_hours)
ax2.set_xticklabels([f"{h}시" for h in all_hours], rotation=45, fontsize=8)
ax2.set_title("유형별 시간대 비교", fontweight="bold")
ax2.set_ylabel("건수")
ax2.legend(fontsize=9)

# 3. 요일별 전체
ax3 = axes[1, 0]
wd_valid = df[df["weekday"].notna()]
if len(wd_valid) > 0:
    wd_counts = wd_valid["weekday"].value_counts().sort_index()
    wd_names = ["월", "화", "수", "목", "금", "토", "일"]
    wd_full = pd.Series([wd_counts.get(i, 0) for i in range(7)], index=wd_names)
    wd_colors = ["#42A5F5"] * 5 + ["#FFCDD2", "#FFCDD2"]
    bars3 = ax3.bar(wd_names, wd_full.values, color=wd_colors, edgecolor="white")
    for bar, val in zip(bars3, wd_full.values):
        if val > 0:
            ax3.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3, str(val), ha="center", fontsize=9)
ax3.set_title("요일별 발신 건수", fontweight="bold")
ax3.set_ylabel("건수")

# 4. 월별 유형 추세
ax4 = axes[1, 1]
for t, color in [("고객콜백", "#2196F3"), ("공급업체", "#4CAF50")]:
    sub = df[df["call_type"] == t].groupby("month").size().reset_index(name="cnt")
    ax4.plot(range(len(sub)), sub["cnt"], "o-", color=color, label=t, linewidth=2, markersize=5)
    ax4.set_xticks(range(len(sub)))
    ax4.set_xticklabels([m[2:] for m in sub["month"]], rotation=45, fontsize=8)
ax4.set_title("월별 유형별 추세", fontweight="bold")
ax4.set_ylabel("건수")
ax4.legend(fontsize=9)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/07_hourly_weekday_pattern.png", dpi=150, bbox_inches="tight")
plt.show()
print("저장: 07_hourly_weekday_pattern.png")

## 6. 고객콜백 — 복잡도 매트릭스 (챗봇 자동화 우선순위)

In [ ]:
if len(cb_df) > 5:
    priority_data = []
    for cat in cb_df["reason_cat"].unique():
        sub = cb_df[cb_df["reason_cat"] == cat]
        if len(sub) == 0:
            continue
        priority_data.append({
            "category": cat,
            "건수": len(sub),
            "평균통화시간": sub["duration_sec"].mean(),
            "평균QA수": sub["qa_count"].mean(),
            "해결률": (sub.get("resolution_status", pd.Series()) == "해결").mean() * 100
                      if "resolution_status" in sub.columns else 0,
        })

    pdata = pd.DataFrame(priority_data)

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.set_facecolor("#F8F9FA")

    x = pdata["건수"].values
    y = 300 - pdata["평균통화시간"].values  # 짧을수록 위 = 자동화 적합
    sizes = (pdata["평균QA수"].values + 0.5) * 250
    priority_score = x / max(x) * 0.6 + (y / max(y + 1)) * 0.4

    sc = ax.scatter(x, y, s=sizes, c=priority_score, cmap="RdYlGn",
                     alpha=0.8, edgecolors="white", linewidth=2, vmin=0, vmax=1)
    for xi, yi, cat in zip(x, y, pdata["category"]):
        ax.annotate(cat, (xi, yi), textcoords="offset points", xytext=(12, 0),
                    fontsize=9, fontweight="bold",
                    arrowprops=dict(arrowstyle="->", color="gray", lw=0.8))

    plt.colorbar(sc, ax=ax, label="챗봇 자동화 우선순위")
    ax.axhline(np.mean(y), color="gray", linestyle="--", linewidth=1, alpha=0.5)
    ax.axvline(np.mean(x), color="gray", linestyle="--", linewidth=1, alpha=0.5)
    ax.text(0.97, 0.97, "TOP PRIORITY\n(빈도↑, 단순)\n→ 챗봇 핵심",
            transform=ax.transAxes, fontsize=9, va="top", ha="right",
            color="darkgreen", fontweight="bold")
    ax.text(0.97, 0.03, "ESCALATE\n(빈도↑, 복잡)\n→ 상담원 연결",
            transform=ax.transAxes, fontsize=9, va="bottom", ha="right", color="red")
    ax.set_xlabel("콜백 건수 (빈도)", fontsize=11)
    ax.set_ylabel("자동화 적합도 (↑ 짧은 통화 = 단순)", fontsize=11)
    ax.set_title("고객콜백 챗봇 자동화 우선순위 매트릭스\n(버블 크기 = 평균 QA 수)",
                 fontsize=13, fontweight="bold")

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/08_chatbot_priority_matrix.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("저장: 08_chatbot_priority_matrix.png")

## 7. 스팟체크 — 대표 사례 검토 (텍스트, 이미지 없음)

In [ ]:
# 고객콜백 — QA가 가장 많은 상담 상위 5건 상세
if len(cb_df) > 0:
    top = cb_df.nlargest(5, "qa_count")
    for _, row in top.iterrows():
        print("=" * 60)
        print(f"파일: {row['file_name']}")
        print(f"통화시간: {row['duration_sec']:.0f}초 | QA: {row['qa_count']}개")
        print(f"콜백이유: {row.get('callback_reason', '')} | 해결: {row.get('resolution_status', '')} | 감정: {row.get('customer_sentiment', '')}")
        print(f"요약: {row['summary']}")
        print()
        for j, qa in enumerate(row["qa_pairs"] or [], 1):
            print(f"  [QA {j}] Q: {qa.get('question', '')}")
            print(f"         A: {qa.get('answer', '')}")
        print()

In [ ]:
# 공급업체 통화 샘플 10건 상세
print("[ 공급업체 통화 샘플 10건 ]")
for _, row in sup_df.head(10).iterrows():
    print("=" * 60)
    print(f"파일: {row['file_name']}")
    print(f"공급사: {row.get('supplier_name', '미확인')} | 통화시간: {row['duration_sec']:.0f}초")
    print(f"요약: {row.get('summary', '')}")
    for p in (row.get("products") or []):
        if isinstance(p, dict):
            stock = "있음" if p.get("stock_available") is True else "없음" if p.get("stock_available") is False else "미확인"
            print(f"  상품: {p.get('product_name','')} | 수량: {p.get('quantity','')} | 단가: {p.get('unit_price','')} | 재고: {stock} | 납기: {p.get('delivery_date','')}")
    confirmed = row.get("order_confirmed")
    confirmed_str = "확정" if confirmed is True else "미확정" if confirmed is False else "미기재"
    print(f"발주: {confirmed_str} | 이슈: {row.get('issues', '') or '없음'}")
    if row.get("follow_up"):
        print(f"후속: {row['follow_up']}")
    print()

## 8. Part A 인사이트 요약

In [ ]:
print("=" * 70)
print("[ Part A — 통화 유형(call_type) 기반 핵심 인사이트 ]")
print("=" * 70)

print(f"""
■ 전체 현황
  총 발신 통화:   {len(df):,}건
  고객콜백:       {len(cb_df):,}건 ({len(cb_df)/len(df)*100:.1f}%)
  공급업체:       {len(sup_df):,}건 ({len(sup_df)/len(df)*100:.1f}%)
  분석 기간:      {df.month.min()} ~ {df.month.max()}
""")

if len(cb_df) > 0:
    top_reason = reason_counts.index[0] if len(reason_counts) > 0 else "(없음)"
    solved_rate = 0
    if "resolution_status" in cb_df.columns:
        solved_rate = (cb_df["resolution_status"].isin(["해결", "부분해결"])).mean() * 100
    sat_rate = 0
    if "customer_sentiment" in cb_df.columns:
        sat_rate = (cb_df["customer_sentiment"] == "만족").mean() * 100
    fu_rate = 0
    if "follow_up_needed" in cb_df.columns:
        fu_rate = (cb_df["follow_up_needed"] == True).mean() * 100

    print(f"""■ 고객콜백 인사이트
  상담 건수:       {len(cb_df):,}건
  가장 많은 콜백 이유: {top_reason}
  해결+부분해결률: {solved_rate:.1f}%
  고객 만족률:     {sat_rate:.1f}%
  후속조치 필요률: {fu_rate:.1f}%
  생성된 QA 쌍:   {len(qa_df):,}개
  통화당 평균 QA: {cb_df["qa_count"].mean():.2f}개
""")

if len(sup_df) > 0:
    confirmed = (sup_df["order_confirmed"] == True).sum()
    issues_cnt = sup_df["issues"].apply(bool).sum()
    no_stock = (prod_df["stock_available"] == False).sum() if len(prod_df) > 0 else 0
    top_prod = prod_df["prod_cat"].value_counts().index[0] if len(prod_df) > 0 else "(없음)"

    print(f"""■ 공급업체 통화 인사이트
  총 공급업체 통화: {len(sup_df):,}건
  발주 확정 건수:   {confirmed}건 ({confirmed/len(sup_df)*100:.1f}%)
  이슈 발생 건수:   {issues_cnt}건 ({issues_cnt/len(sup_df)*100:.1f}%)
  언급 상품 총수:   {len(prod_df):,}건
  재고 없음 건수:   {no_stock}건
  가장 많이 언급된 상품: {top_prod}
""")

print("""■ 챗봇 시나리오 적용 포인트
  고객콜백 → 이전 문의 연속성 파악 후 답변 제공
  가장 많은 콜백 주제 우선 자동화 권장
  미해결 건은 상담원 에스컬레이션 필수
  공급업체 이슈(재고없음/납기지연)는 즉시 고객 안내 시나리오 필요
""")

---
# Part B. 통화 목적 · 수신빈도 기반 분석

`call_type`(고객콜백/공급업체/기타) 분류와 별개로, 전사문·요약 키워드로 재분류한 **통화 목적(topic)**과
**수신번호별 연락 빈도**를 축으로 분석합니다. 어떤 번호와 왜 자주 연락하는지 같은, `call_type` 분류만으로는
드러나지 않는 실질적 업무 패턴을 보완적으로 확인하기 위한 관점입니다.

## 9. 통화 목적 재분류 + 데이터 준비

In [ ]:
# 통화 목적 분류 키워드
TOPIC_KEYWORDS = {
    "인쇄/제작 확인": ["인쇄", "제작", "시안", "디자인", "색상", "로고", "파일", "작업", "컬러", "이미지"],
    "납기/배송 확인": ["납기", "출고", "발송", "언제", "도착", "배송", "당일", "다음주", "내일", "빨리", "급해", "선적"],
    "재고/수량 확인": ["재고", "수량", "가능", "있나요", "몇 개", "품절", "물량"],
    "가격/단가 협의": ["가격", "단가", "얼마", "견적", "비용", "인상", "단가표", "개당"],
    "주문/발주 처리": ["주문", "발주", "오더", "주문서", "진행"],
    "상품 사양 문의": ["사양", "규격", "크기", "사이즈", "소재", "재질"],
}


def classify_topic(row):
    text = (row.get("transcript") or "") + " " + (row.get("summary") or "")
    scores = {}
    for topic, kws in TOPIC_KEYWORDS.items():
        score = sum(text.count(kw) for kw in kws)
        if score > 0:
            scores[topic] = score
    return max(scores, key=scores.get) if scores else "기타"


df["topic"] = df.apply(classify_topic, axis=1)

# 수신자별 통화 빈도
receiver_freq = df["receiver"].value_counts()
df["receiver_freq"] = df["receiver"].map(receiver_freq)


def freq_band(n):
    if n == 1:
        return "1회 (일회성)"
    if n == 2:
        return "2회"
    if n <= 5:
        return "3~5회"
    if n <= 10:
        return "6~10회"
    if n <= 20:
        return "11~20회"
    return "21회+ (핵심 연락처)"


df["receiver_band"] = df["receiver_freq"].apply(freq_band)

topic_counts = df["topic"].value_counts()
topic_order_exist = [t for t in TOPIC_ORDER if t in topic_counts.index]
total = len(df)

print("=" * 55)
print("통화 목적 재분류 완료")
print("=" * 55)
print(f"전체 발신통화: {total:,}건")
print(f"고유 수신자:   {df.receiver.nunique():,}명")
print(f"평균 통화시간: {df.duration_sec.mean():.1f}초")
print()
print("[ 통화 목적별 분포 ]")
for t, cnt in topic_counts.items():
    bar = "█" * int(cnt / total * 30)
    print(f"  {t:<12}: {cnt:>4}건 ({cnt/total*100:5.1f}%)  {bar}")

## 10. 통화 목적별 분포 & 통화시간

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("발신통화 목적별 분포 & 통화시간 분석", fontsize=15, fontweight="bold")

tc_vals = [topic_counts.get(t, 0) for t in topic_order_exist]
tc_colors = [TOPIC_COLORS[t] for t in topic_order_exist]

# 1. 수평 막대 (건수 + %)
ax1 = axes[0]
bars1 = ax1.barh(topic_order_exist[::-1], tc_vals[::-1],
                  color=tc_colors[::-1], alpha=0.85, edgecolor="white")
for bar, val in zip(bars1, tc_vals[::-1]):
    ax1.text(bar.get_width() + 3, bar.get_y() + bar.get_height() / 2,
              f"{val}건 ({val/total*100:.1f}%)", va="center", fontsize=9)
ax1.set_xlim(0, max(tc_vals) * 1.35)
ax1.set_title("통화 목적별 건수", fontweight="bold")
ax1.set_xlabel("건수")
ax1.tick_params(axis="y", labelsize=9)

# 2. 목적별 평균 통화시간
ax2 = axes[1]
topic_dur = df.groupby("topic")["duration_sec"].mean().reindex(topic_order_exist)
bars2 = ax2.barh(topic_order_exist[::-1], topic_dur.values[::-1],
                  color=tc_colors[::-1], alpha=0.85, edgecolor="white")
ax2.axvline(df["duration_sec"].mean(), color="red", linestyle="--", linewidth=1.5,
            label=f"전체 평균 {df['duration_sec'].mean():.0f}초")
for bar, val in zip(bars2, topic_dur.values[::-1]):
    ax2.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
              f"{val:.0f}초", va="center", fontsize=9)
ax2.set_title("목적별 평균 통화시간", fontweight="bold")
ax2.set_xlabel("초")
ax2.legend(fontsize=8)
ax2.tick_params(axis="y", labelsize=9)

# 3. 목적별 통화시간 박스플롯
ax3 = axes[2]
data_box = [df[df["topic"] == t]["duration_sec"].clip(0, 600).values for t in topic_order_exist]
bp = ax3.boxplot(data_box, labels=[t.replace(" ", "\n") for t in topic_order_exist],
                  patch_artist=True, medianprops=dict(color="black", linewidth=2))
for patch, color in zip(bp["boxes"], tc_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax3.set_title("목적별 통화시간 분포 (600초 상한)", fontweight="bold")
ax3.set_ylabel("통화시간 (초)")
ax3.tick_params(axis="x", labelsize=7)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/09_topic_distribution_duration.png", dpi=150, bbox_inches="tight")
plt.show()
print("저장: 09_topic_distribution_duration.png")

## 11. 통화 목적별 월별 트렌드

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 11))

monthly_topic = df.groupby(["month", "topic"]).size().unstack(fill_value=0)
monthly_topic = monthly_topic.reindex(
    columns=[t for t in topic_order_exist if t in monthly_topic.columns], fill_value=0
)

# 1. 누적 막대 (월별 목적 건수)
ax1 = axes[0]
bottom = np.zeros(len(monthly_topic))
for t in monthly_topic.columns:
    vals = monthly_topic[t].values
    ax1.bar(range(len(monthly_topic)), vals, bottom=bottom,
            label=t, color=TOPIC_COLORS[t], alpha=0.85, edgecolor="white", linewidth=0.5)
    bottom += vals
ax1.set_xticks(range(len(monthly_topic)))
ax1.set_xticklabels([m[2:] for m in monthly_topic.index], rotation=45, fontsize=8)
ax1.set_title("월별 통화 목적 건수 (누적)", fontweight="bold", fontsize=13)
ax1.set_ylabel("건수")
ax1.legend(loc="upper left", fontsize=8, ncol=2)

# 2. 히트맵 (비율)
ax2 = axes[1]
monthly_pct = monthly_topic.div(monthly_topic.sum(axis=1), axis=0) * 100
monthly_pct.columns.name = None
monthly_pct.index = [m[2:] for m in monthly_pct.index]
sns.heatmap(
    monthly_pct.T, ax=ax2, annot=True, fmt=".0f", cmap="YlOrRd",
    linewidths=0.5, linecolor="white", cbar_kws={"label": "비율(%)"},
    annot_kws={"size": 8},
)
ax2.set_title("월별 통화 목적 비율 히트맵 (%)", fontweight="bold", fontsize=13)
ax2.set_xlabel("연월")
ax2.set_ylabel("통화 목적")
ax2.tick_params(axis="x", rotation=45, labelsize=7)
ax2.tick_params(axis="y", rotation=0, labelsize=9)

plt.tight_layout(pad=3)
plt.savefig(f"{OUTPUT_DIR}/10_topic_monthly_trend.png", dpi=150, bbox_inches="tight")
plt.show()
print("저장: 10_topic_monthly_trend.png")

## 12. 수신자 연락 빈도 분석

In [ ]:
BAND_ORDER = ["1회 (일회성)", "2회", "3~5회", "6~10회", "11~20회", "21회+ (핵심 연락처)"]
BAND_COLORS = ["#CFD8DC", "#B0BEC5", "#90CAF9", "#42A5F5", "#1565C0", "#E91E63"]

recv_agg = df.groupby("receiver").agg(
    통화건수=("duration_sec", "count"),
    평균통화시간=("duration_sec", "mean"),
    총통화시간=("duration_sec", "sum"),
    빈도구간=("receiver_band", "first"),
    주요목적=("topic", lambda x: x.value_counts().index[0]),
).reset_index()

band_counts_recv = recv_agg["빈도구간"].value_counts().reindex(BAND_ORDER, fill_value=0)
band_call_counts = df.groupby("receiver_band").size().reindex(BAND_ORDER, fill_value=0)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("수신자 연락 빈도 분석", fontsize=15, fontweight="bold")

# 1. 수신자 수 by 빈도구간
ax1 = axes[0]
bars1 = ax1.bar(BAND_ORDER, band_counts_recv.values, color=BAND_COLORS, edgecolor="white", alpha=0.9)
for bar, val in zip(bars1, band_counts_recv.values):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
              f"{val}명", ha="center", fontsize=9, fontweight="bold")
total_recv = recv_agg.shape[0]
ax1.set_title(f"연락 빈도별 수신자 수 (총 {total_recv}명)", fontweight="bold")
ax1.set_ylabel("수신자 수 (명)")
ax1.tick_params(axis="x", rotation=30, labelsize=8)

# 2. 통화 건수 기여 by 빈도구간
ax2 = axes[1]
bars2 = ax2.bar(BAND_ORDER, band_call_counts.values, color=BAND_COLORS, edgecolor="white", alpha=0.9)
for bar, val in zip(bars2, band_call_counts.values):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
              f"{val}건\n({val/total*100:.0f}%)", ha="center", fontsize=8, fontweight="bold")
ax2.set_title("빈도 구간별 통화 건수 기여", fontweight="bold")
ax2.set_ylabel("통화 건수")
ax2.tick_params(axis="x", rotation=30, labelsize=8)

# 3. 핵심 연락처 Top20
ax3 = axes[2]
top20 = recv_agg.nlargest(20, "통화건수")
y_pos = range(len(top20))
bar_colors3 = [TOPIC_COLORS.get(t, "#9E9E9E") for t in top20["주요목적"]]
bars3 = ax3.barh(list(y_pos), top20["통화건수"].tolist(), color=bar_colors3, edgecolor="white", alpha=0.85)
ax3.set_yticks(list(y_pos))
# 번호 마스킹: 앞 4자리만 표시
ax3.set_yticklabels([f"{r[:4]}****" for r in top20["receiver"]], fontsize=8)
for bar, val in zip(bars3, top20["통화건수"]):
    ax3.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
              f"{val}건", va="center", fontsize=8)
ax3.set_title("자주 연락한 상위 20 수신자\n(색상 = 주요 통화 목적)", fontweight="bold")
ax3.set_xlabel("통화 건수")
legend_patches = [mpatches.Patch(color=TOPIC_COLORS[t], label=t) for t in topic_order_exist]
ax3.legend(handles=legend_patches, fontsize=6, loc="lower right", ncol=1)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/11_receiver_frequency.png", dpi=150, bbox_inches="tight")
plt.show()
print("저장: 11_receiver_frequency.png")

print("\n[ 빈도 구간별 요약 ]")
for band, recv_n, call_n in zip(BAND_ORDER, band_counts_recv.values, band_call_counts.values):
    print(f"  {band:<18}: 수신자 {recv_n:>3}명 → 통화 {call_n:>4}건 ({call_n/total*100:.0f}%)")

## 13. 핵심 연락처의 통화 목적 패턴

In [ ]:
# 6회 이상 연락한 수신자만 분석 (핵심 연락처)
core_receivers = recv_agg[recv_agg["통화건수"] >= 6].nlargest(15, "통화건수")["receiver"].tolist()
core_df = df[df["receiver"].isin(core_receivers)].copy()

core_pivot = core_df.groupby(["receiver", "topic"]).size().unstack(fill_value=0)
core_pivot = core_pivot.reindex(columns=[t for t in topic_order_exist if t in core_pivot.columns], fill_value=0)
core_pivot = core_pivot.loc[core_pivot.sum(axis=1).sort_values(ascending=False).index]
core_pivot.index = [f"{r[:4]}****" for r in core_pivot.index]

if len(core_pivot) == 0:
    print("핵심 연락처(6회 이상) 없음")
else:
    fig, axes = plt.subplots(1, 2, figsize=(17, 7))
    fig.suptitle("핵심 연락처 통화 목적 패턴 (6회 이상, 상위 15개)", fontsize=14, fontweight="bold")

    ax1 = axes[0]
    bottom = np.zeros(len(core_pivot))
    for t in core_pivot.columns:
        vals = core_pivot[t].values
        ax1.barh(range(len(core_pivot)), vals, left=bottom,
                 label=t, color=TOPIC_COLORS.get(t, "#9E9E9E"), alpha=0.85, edgecolor="white")
        bottom += vals
    ax1.set_yticks(range(len(core_pivot)))
    ax1.set_yticklabels(core_pivot.index, fontsize=9)
    ax1.set_title("핵심 수신자별 통화 목적 건수", fontweight="bold")
    ax1.set_xlabel("통화 건수")
    ax1.legend(loc="lower right", fontsize=7)

    ax2 = axes[1]
    core_pct = core_pivot.div(core_pivot.sum(axis=1), axis=0) * 100
    sns.heatmap(
        core_pct, ax=ax2, annot=True, fmt=".0f", cmap="Blues",
        linewidths=0.5, linecolor="white", cbar_kws={"label": "비율(%)"},
        annot_kws={"size": 9},
    )
    ax2.set_title("핵심 수신자별 통화 목적 비율 (%)", fontweight="bold")
    ax2.set_xlabel("통화 목적")
    ax2.set_ylabel("")
    ax2.tick_params(axis="x", rotation=25, labelsize=8)
    ax2.tick_params(axis="y", rotation=0, labelsize=9)

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/12_core_receiver_topic_pattern.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("저장: 12_core_receiver_topic_pattern.png")

## 14. 시간대 & 요일별 발신 패턴 (목적별)

In [ ]:
hour_valid = df[df["hour"].notna()].copy()
wd_valid = df[df["weekday"].notna()].copy()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("시간대 & 요일별 발신 패턴 (통화 목적별)", fontsize=14, fontweight="bold")

all_hours = list(range(8, 20))
wd_names = ["월", "화", "수", "목", "금", "토", "일"]

# 1. 시간대별 건수 (목적별 누적)
ax1 = axes[0, 0]
if len(hour_valid) > 0:
    ht = hour_valid.groupby(["hour", "topic"]).size().unstack(fill_value=0)
    ht = ht.reindex(index=all_hours, fill_value=0)
    ht = ht.reindex(columns=[t for t in topic_order_exist if t in ht.columns], fill_value=0)
    bot = np.zeros(len(all_hours))
    for t in ht.columns:
        ax1.bar([f"{h}시" for h in all_hours], ht[t].values, bottom=bot,
                color=TOPIC_COLORS.get(t, "#9E9E9E"), alpha=0.85, edgecolor="white", label=t)
        bot += ht[t].values
    ax1.legend(fontsize=7, ncol=2)
ax1.set_title("시간대별 발신 건수", fontweight="bold")
ax1.set_ylabel("건수")
ax1.tick_params(axis="x", rotation=45, labelsize=8)

# 2. 요일별 건수 (목적별 누적)
ax2 = axes[0, 1]
if len(wd_valid) > 0:
    wt = wd_valid.groupby(["weekday", "topic"]).size().unstack(fill_value=0)
    wt = wt.reindex(index=range(7), fill_value=0)
    wt = wt.reindex(columns=[t for t in topic_order_exist if t in wt.columns], fill_value=0)
    bot2 = np.zeros(7)
    for t in wt.columns:
        ax2.bar(wd_names, wt[t].values, bottom=bot2,
                color=TOPIC_COLORS.get(t, "#9E9E9E"), alpha=0.85, edgecolor="white", label=t)
        bot2 += wt[t].values
    ax2.legend(fontsize=7, ncol=2)
ax2.set_title("요일별 발신 건수", fontweight="bold")
ax2.set_ylabel("건수")

# 3. 시간대별 평균 통화시간
ax3 = axes[1, 0]
if len(hour_valid) > 0:
    hour_dur = hour_valid.groupby("hour")["duration_sec"].mean().reindex(all_hours)
    color_h = [
        "#FFCDD2" if h < 9 or h >= 18 else
        "#90CAF9" if h < 12 else
        "#A5D6A7" if h < 14 else "#42A5F5"
        for h in all_hours
    ]
    bars_h = ax3.bar([f"{h}시" for h in all_hours], hour_dur.values, color=color_h, edgecolor="white")
    ax3.axhline(df["duration_sec"].mean(), color="red", linestyle="--", linewidth=1.5,
                label=f"전체 평균 {df['duration_sec'].mean():.0f}초")
    for bar, val in zip(bars_h, hour_dur.values):
        if not np.isnan(val) and val > 0:
            ax3.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                      f"{val:.0f}", ha="center", fontsize=7)
    ax3.legend(fontsize=8)
ax3.set_title("시간대별 평균 통화시간 (초)", fontweight="bold")
ax3.set_ylabel("초")
ax3.tick_params(axis="x", rotation=45, labelsize=8)

# 4. 요일별 평균 통화시간
ax4 = axes[1, 1]
if len(wd_valid) > 0:
    wd_dur = wd_valid.groupby("weekday")["duration_sec"].mean().reindex(range(7))
    color_wd = ["#42A5F5"] * 5 + ["#FFCDD2", "#FFCDD2"]
    bars_wd = ax4.bar(wd_names, wd_dur.values, color=color_wd, edgecolor="white")
    ax4.axhline(df["duration_sec"].mean(), color="red", linestyle="--", linewidth=1.5,
                label=f"전체 평균 {df['duration_sec'].mean():.0f}초")
    for bar, val in zip(bars_wd, wd_dur.values):
        if not np.isnan(val) and val > 0:
            ax4.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                      f"{val:.0f}", ha="center", fontsize=9)
    ax4.legend(fontsize=8)
ax4.set_title("요일별 평균 통화시간 (초)", fontweight="bold")
ax4.set_ylabel("초")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/13_hourly_weekday_by_topic.png", dpi=150, bbox_inches="tight")
plt.show()
print("저장: 13_hourly_weekday_by_topic.png")

## 15. 목적별 핵심 키워드 비교

In [ ]:
# 목적별 전사문에서 키워드 빈도 추출
DETAIL_KWS = [
    "인쇄", "제작", "시안", "디자인", "색상", "로고", "파일",
    "납기", "출고", "발송", "언제", "당일", "내일", "빨리",
    "재고", "수량", "가능", "있나요", "몇 개", "품절",
    "가격", "단가", "얼마", "견적", "비용", "개당",
    "주문", "발주", "오더", "주문서",
    "연락", "확인", "감사", "안녕",
]

topic_kw_matrix = {}
for t in topic_order_exist:
    text = " ".join(df[df["topic"] == t]["transcript"].dropna().astype(str))
    n_calls = max((df["topic"] == t).sum(), 1)
    topic_kw_matrix[t] = {kw: text.count(kw) / n_calls * 100 for kw in DETAIL_KWS}

kw_df = pd.DataFrame(topic_kw_matrix).fillna(0)
kw_df["variance"] = kw_df.var(axis=1)
top_kws = kw_df.nlargest(20, "variance").drop(columns="variance")

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle("통화 목적별 핵심 키워드 비교", fontsize=14, fontweight="bold")

# 1. 히트맵 (100건당 출현)
ax1 = axes[0]
sns.heatmap(
    top_kws, ax=ax1, annot=True, fmt=".1f", cmap="YlOrRd",
    linewidths=0.5, linecolor="white", cbar_kws={"label": "100건당 출현"},
    annot_kws={"size": 8},
)
ax1.set_title("목적별 키워드 밀도 히트맵\n(목적 간 차이가 큰 키워드 Top 20)", fontweight="bold")
ax1.set_xlabel("통화 목적")
ax1.set_ylabel("키워드")
ax1.tick_params(axis="x", rotation=25, labelsize=8)
ax1.tick_params(axis="y", rotation=0, labelsize=9)

# 2. 목적별 대표 키워드
ax2 = axes[1]
top5_per_topic = {}
for t in topic_order_exist:
    if t in top_kws.columns:
        col = top_kws[t]
        other_max = top_kws.drop(columns=t).max(axis=1)
        diff = col - other_max
        top5_per_topic[t] = diff.nlargest(4).index.tolist()

y_base = 0
tick_positions = []
tick_labels = []
for t, kws in top5_per_topic.items():
    vals = [topic_kw_matrix[t].get(kw, 0) for kw in kws]
    positions = [y_base + i for i in range(len(kws))]
    ax2.barh(positions, vals, color=TOPIC_COLORS.get(t, "#9E9E9E"),
              alpha=0.85, edgecolor="white", label=t)
    for pos, kw, val in zip(positions, kws, vals):
        tick_positions.append(pos)
        tick_labels.append(kw)
        ax2.text(val + 0.2, pos, f"{val:.1f}", va="center", fontsize=7)
    y_base += len(kws) + 1

ax2.set_yticks(tick_positions)
ax2.set_yticklabels(tick_labels, fontsize=8)
ax2.set_title("목적별 특징 키워드\n(다른 목적 대비 더 많이 나오는 키워드)", fontweight="bold")
ax2.set_xlabel("100건당 출현 횟수")
ax2.legend(fontsize=7, loc="lower right")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/14_topic_keyword_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("저장: 14_topic_keyword_comparison.png")

## 16. 목적별 상품 연관 분석

In [ ]:
PROD_CATS = {
    "달력/캘린더": ["달력", "캘린더", "탁상형", "탁상달력"],
    "메모지/노트": ["메모지", "메모", "노트", "수첩", "다이어리"],
    "볼펜/펜": ["볼펜", "펜", "사인펜"],
    "가방/에코백": ["가방", "에코백", "파우치", "백팩", "토트"],
    "머그컵/텀블러": ["머그", "텀블러", "컵"],
    "케이스": ["케이스", "하드케이스"],
    "우산": ["우산", "양산"],
    "USB/전자": ["usb", "USB", "보조배터리"],
    "수건/타올": ["수건", "타올", "타월"],
    "인쇄물": ["인쇄물", "브로셔", "명함", "스티커"],
}


def classify_product_topic(name):
    name_lower = str(name).lower()
    for cat, kws in PROD_CATS.items():
        if any(kw.lower() in name_lower for kw in kws):
            return cat
    return "기타"


all_prods_topic = []
for _, row in df.iterrows():
    for p in (row.get("products") or []):
        if isinstance(p, dict) and p.get("product_name"):
            all_prods_topic.append({
                "topic": row["topic"],
                "month": row["month"],
                "product_cat": classify_product_topic(p["product_name"]),
                "product_name": p["product_name"],
                "quantity": p.get("quantity"),
                "unit_price": p.get("unit_price"),
            })
prod_df_topic = pd.DataFrame(all_prods_topic)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("통화 목적별 상품 연관 분석", fontsize=14, fontweight="bold")

ax1 = axes[0]
if len(prod_df_topic) > 0:
    prod_cross = pd.crosstab(prod_df_topic["product_cat"], prod_df_topic["topic"])
    prod_cross = prod_cross.reindex(
        columns=[t for t in topic_order_exist if t in prod_cross.columns], fill_value=0
    )
    sns.heatmap(prod_cross, ax=ax1, annot=True, fmt="d", cmap="Blues",
                linewidths=0.5, linecolor="white", cbar_kws={"label": "건수"},
                annot_kws={"size": 9})
    ax1.set_title("상품 카테고리 × 통화 목적", fontweight="bold")
    ax1.set_xlabel("통화 목적")
    ax1.set_ylabel("상품 카테고리")
    ax1.tick_params(axis="x", rotation=25, labelsize=8)
    ax1.tick_params(axis="y", rotation=0)

ax2 = axes[1]
if len(prod_df_topic) > 0:
    prod_pct = prod_cross.div(prod_cross.sum(axis=0), axis=1) * 100
    prod_cats_exist = prod_pct.index.tolist()
    colors_pc = plt.cm.Set3(np.linspace(0, 1, len(prod_cats_exist)))
    bottom_p = np.zeros(len(prod_pct.columns))
    for cat, color in zip(prod_cats_exist, colors_pc):
        vals_p = prod_pct.loc[cat].values if cat in prod_pct.index else np.zeros(len(prod_pct.columns))
        ax2.bar(prod_pct.columns, vals_p, bottom=bottom_p, color=color,
                edgecolor="white", label=cat, alpha=0.85)
        for xi, (b, v) in enumerate(zip(bottom_p, vals_p)):
            if v > 8:
                ax2.text(xi, b + v / 2, f"{v:.0f}%", ha="center", va="center", fontsize=7)
        bottom_p += vals_p
    ax2.set_title("통화 목적별 상품 구성 비율 (%)", fontweight="bold")
    ax2.set_ylabel("비율 (%)")
    ax2.set_ylim(0, 115)
    ax2.tick_params(axis="x", rotation=25, labelsize=8)
    ax2.legend(fontsize=7, loc="upper right", ncol=2, bbox_to_anchor=(1.0, 1.12))
else:
    ax1.text(0.5, 0.5, "상품 데이터 없음", ha="center", va="center", transform=ax1.transAxes)
    ax2.text(0.5, 0.5, "상품 데이터 없음", ha="center", va="center", transform=ax2.transAxes)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/15_topic_product_association.png", dpi=150, bbox_inches="tight")
plt.show()
print("저장: 15_topic_product_association.png")

## 17. 종합 운영 인사이트

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle("발신통화 종합 운영 인사이트", fontsize=15, fontweight="bold")

# 1. 통화 목적 × 수신자 빈도 구간 교차
ax1 = axes[0, 0]
band_topic = df.groupby(["receiver_band", "topic"]).size().unstack(fill_value=0)
band_topic = band_topic.reindex(index=BAND_ORDER, fill_value=0)
band_topic = band_topic.reindex(columns=[t for t in topic_order_exist if t in band_topic.columns], fill_value=0)
band_topic_pct = band_topic.div(band_topic.sum(axis=1), axis=0) * 100
bottom_bt = np.zeros(len(BAND_ORDER))
for t in band_topic_pct.columns:
    vals_bt = band_topic_pct[t].values
    ax1.bar(range(len(BAND_ORDER)), vals_bt, bottom=bottom_bt,
            color=TOPIC_COLORS.get(t, "#9E9E9E"), alpha=0.85, edgecolor="white", label=t)
    for xi, (b, v) in enumerate(zip(bottom_bt, vals_bt)):
        if v > 12:
            ax1.text(xi, b + v / 2, f"{v:.0f}%", ha="center", va="center",
                      fontsize=7, color="white", fontweight="bold")
    bottom_bt += vals_bt
ax1.set_xticks(range(len(BAND_ORDER)))
ax1.set_xticklabels([b.replace(" (", "\n(") for b in BAND_ORDER], fontsize=7, rotation=20)
ax1.set_title("연락 빈도별 통화 목적 구성\n(일회성 vs 핵심 연락처)", fontweight="bold")
ax1.set_ylabel("비율 (%)")
ax1.legend(fontsize=7, loc="upper right", ncol=1)

# 2. 목적별 통화시간 구간 분포
ax2 = axes[0, 1]
dur_bins = [0, 30, 60, 120, 300, float("inf")]
dur_labels = ["~30초", "30~60초", "1~2분", "2~5분", "5분+"]
df["dur_band2"] = pd.cut(df["duration_sec"], bins=dur_bins, labels=dur_labels)
topic_dur_dist = df.groupby(["topic", "dur_band2"]).size().unstack(fill_value=0)
topic_dur_dist = topic_dur_dist.reindex(
    index=[t for t in topic_order_exist if t in topic_dur_dist.index], fill_value=0
)
topic_dur_pct = topic_dur_dist.div(topic_dur_dist.sum(axis=1), axis=0) * 100
dur_colors2 = ["#FFCDD2", "#FFCC80", "#A5D6A7", "#90CAF9", "#1565C0"]
bottom_td = np.zeros(len(topic_dur_pct))
for band, color in zip(dur_labels, dur_colors2):
    if band in topic_dur_pct.columns:
        vals_td = topic_dur_pct[band].values
        ax2.barh(range(len(topic_dur_pct)), vals_td, left=bottom_td,
                 color=color, edgecolor="white", label=band, alpha=0.9)
        for xi, (b, v) in enumerate(zip(bottom_td, vals_td)):
            if v > 10:
                ax2.text(b + v / 2, xi, f"{v:.0f}%", ha="center", va="center",
                          fontsize=8, color="black")
        bottom_td += vals_td
ax2.set_yticks(range(len(topic_dur_pct)))
ax2.set_yticklabels(topic_dur_pct.index, fontsize=9)
ax2.set_title("통화 목적별 통화시간 구간 비율", fontweight="bold")
ax2.set_xlabel("비율 (%)")
ax2.legend(fontsize=8, loc="lower right")

# 3. 월별 총 발신 건수 + 이동평균
ax3 = axes[1, 0]
monthly_total = df.groupby("month").size().reset_index(name="건수")
x3 = range(len(monthly_total))
ax3.bar(x3, monthly_total["건수"], color="#90CAF9", alpha=0.7, edgecolor="white", label="월별 건수")
if len(monthly_total) >= 3:
    ma3 = monthly_total["건수"].rolling(3, center=True).mean()
    ax3.plot(list(x3), ma3, "r-o", linewidth=2, markersize=4, label="3개월 이동평균")
ax3.set_xticks(list(x3))
ax3.set_xticklabels([m[2:] for m in monthly_total["month"]], rotation=45, fontsize=7)
ax3.set_title("월별 발신 건수 추세 (3개월 이동평균)", fontweight="bold")
ax3.set_ylabel("건수")
ax3.legend(fontsize=8)

# 4. 핵심 지표 요약 텍스트 박스
ax4 = axes[1, 1]
ax4.axis("off")
core_21plus = (df["receiver_band"] == "21회+ (핵심 연락처)").sum()
top_topic = topic_counts.index[0]
top_topic_pct = topic_counts.iloc[0] / total * 100
one_time_pct = (df["receiver_band"] == "1회 (일회성)").sum() / total * 100
peak_hour = df[df["hour"].notna()].groupby("hour").size().idxmax() if df["hour"].notna().sum() > 0 else "-"

summary_text = (
    f"발신통화 핵심 요약\n"
    f"{'━'*30}\n\n"
    f"▶ 총 발신: {total:,}건 / {df.receiver.nunique():,}개 번호\n\n"
    f"▶ 가장 많은 통화 목적\n"
    f"   {top_topic} ({top_topic_pct:.0f}%)\n\n"
    f"▶ 일회성 연락처 비중: {one_time_pct:.0f}%\n\n"
    f"▶ 핵심 연락처(21회+): {df[df['receiver_band']=='21회+ (핵심 연락처)'].receiver.nunique()}개 번호\n"
    f"   → 전체 건수의 {core_21plus/total*100:.0f}% 집중\n\n"
    f"▶ 피크 발신 시간대: {peak_hour}시\n\n"
    f"▶ 평균 통화시간: {df.duration_sec.mean():.0f}초\n"
    f"   (중앙값: {df.duration_sec.median():.0f}초)"
)
ax4.text(0.05, 0.95, summary_text, transform=ax4.transAxes,
         fontsize=11, va="top", ha="left",
         bbox=dict(boxstyle="round,pad=0.8", facecolor="#E3F2FD", alpha=0.8),
         linespacing=1.6)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/16_comprehensive_insight.png", dpi=150, bbox_inches="tight")
plt.show()
print("저장: 16_comprehensive_insight.png")

---
## 저장 확인

In [ ]:
import glob

saved = sorted(glob.glob(f"{OUTPUT_DIR}/*.png"))
print(f"저장된 분석 이미지 ({len(saved)}개):")
for f in saved:
    print(f"  {f}")